# 01 - Data exploration

Set `DEMO = False` to explore your real data in `data/raw/`. With `DEMO = True` this uses the **SYNTHETIC** demo dataset (software testing only - not project data).

Run from `notebooks/` with the project's Python environment.

In [ ]:
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt
from src.config import load_config
from src.preprocessing import load_dataset

DEMO = True
cfg = load_config(demo=DEMO)
df, report = load_dataset(cfg)
print('DEMO (synthetic) data' if DEMO else 'REAL data')
print(report.as_dict())
df.head()

## Class balance and per-session summary
Readings inside one session are strongly correlated, so look at the number of *sessions*, not just rows.

In [ ]:
print(df['irrigation_label'].value_counts())
df.groupby('session_id').agg(rows=('soil_raw', 'size'), soil_min=('soil_raw', 'min'), soil_max=('soil_raw', 'max'),
                             water_share=('irrigation_label', lambda s: (s == 'WATER').mean()))

## Does the raw sensor value rise or fall with wetness?
Compare against what you measured during calibration (dry vs wet). Do not assume a direction.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for s, g in df.groupby('session_id'):
    ax.plot(g['timestamp'], g['soil_raw'], lw=1)
ax.set_ylabel('soil_raw (ADC)'); ax.set_title('Raw soil signal per session'); plt.xticks(rotation=30); plt.show()

## Features vs label

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, cfg.feature_names):
    for lab, g in df.groupby('irrigation_label'):
        ax.hist(g[col], bins=25, alpha=0.6, label=lab)
    ax.set_xlabel(col)
axes[0].legend(); plt.tight_layout(); plt.show()

## Correlation between inputs
DHT11 temperature/humidity are usually anti-correlated; strong correlation limits what they add beyond each other.

In [ ]:
df[cfg.feature_names].corr().round(2)